# KuaiRand 指标体系与业务概览

## 分析目标

本阶段从业务视角建立短视频推荐场景的核心指标体系，并基于同期数据比较正常推荐与随机曝光的整体表现。

本 Notebook 重点回答：

1. 短视频推荐业务应该关注哪些核心指标？
2. 正常推荐与随机曝光在规模、消费、互动和风险指标上有什么差异？
3. 哪些指标适合作为后续 A/B 实验和因果分析的核心指标？
4. 当前对比结果属于什么类型的结论，不能过度解释成什么？


## 1. 业务目标与用户行为漏斗

本项目的业务目标可以概括为：

> 推荐机制是否能帮助用户看到感兴趣的视频，产生更深度的观看和正向互动，同时避免负反馈上升？

短视频推荐场景中的用户行为路径可以理解为：

```text
曝光 → 有效播放反馈 → 长播 → 互动 → 负反馈
```

其中：

- **曝光**：用户看到一个视频推荐；
- **有效播放反馈**：用户对曝光产生了有效消费行为；
- **长播**：用户观看达到平台定义的长观看标准；
- **互动**：用户点赞、评论、关注、转发等；
- **负反馈**：用户表达不喜欢或反感。

指标体系就是基于这个行为漏斗，为每个关键环节定义可衡量的指标。


## 2. 指标体系设计

| 层级 | 指标名称 | 计算口径 | 业务含义 |
|---|---|---|---|
| 规模 | 曝光量 | 日志记录数 | 推荐产生的总曝光规模 |
| 规模 | 活跃用户数 | `user_id` 去重数 | 被推荐覆盖的用户规模 |
| 规模 | 曝光视频数 | `video_id` 去重数 | 被推荐覆盖的视频规模 |
| 规模 | 人均曝光次数 | 曝光量 / 活跃用户数 | 每个用户平均看到多少视频 |
| 消费 | 有效播放反馈率 | `is_click=1` 的曝光数 / 总曝光数 | 曝光后是否产生有效消费反馈 |
| 消费 | 长播率 | `long_view=1` 的曝光数 / 总曝光数 | 用户是否产生深度观看 |
| 消费 | 平均播放时长 | `play_time_ms` 平均值 | 单次曝光平均观看时长 |
| 消费 | 中位数播放比例 | `play_time_ms / duration_ms` 的中位数 | 用户通常看完视频的比例 |
| 互动 | 点赞率 | `is_like=1` 的曝光数 / 总曝光数 | 正向反馈效率 |
| 互动 | 评论率 | `is_comment=1` 的曝光数 / 总曝光数 | 深度参与程度 |
| 互动 | 关注率 | `is_follow=1` 的曝光数 / 总曝光数 | 对创作者产生兴趣 |
| 互动 | 转发率 | `is_forward=1` 的曝光数 / 总曝光数 | 内容传播能力 |
| 风险 | 负反馈率 | `is_hate=1` 的曝光数 / 总曝光数 | 用户反感程度 |
| 数据质量 | 零视频时长占比 | `duration_ms=0` 的曝光数 / 总曝光数 | 播放比例相关指标的可用性 |

说明：

- `is_click` 在 KuaiRand 中不一定总是字面意义的点击，和页面形态有关，因此这里命名为“有效播放反馈率”更严谨；
- `long_view` 是官方提供的长播标签：短视频需基本看完，长视频观看达到一定时长即可算长播；
- 互动类指标本阶段统一以曝光次数为分母，用于衡量“每次推荐曝光带来的反馈效率”。


## 3. 数据读取与最小清洗规则

本阶段只比较同期数据：

- 随机曝光日志：`2022-04-22` 至 `2022-05-08`
- 正常推荐日志：`2022-04-22` 至 `2022-05-08`

不使用前期正常推荐日志进行直接对比，因为时间窗口不同。

沿用第一阶段的数据质量处理原则：

- 删除完全重复记录，每组保留一条；
- 保留 `duration_ms=0` 的曝光记录；
- 计算播放比例时只使用 `duration_ms>0` 的记录；
- 不修改原始字段，只新增派生字段。


In [5]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data/raw/KuaiRand-Pure/data"

print("项目目录：", PROJECT_ROOT)
print("数据目录存在：", DATA_DIR.exists())


项目目录： /Users/nora/Documents/GitHub/kuairand-product-analytics
数据目录存在： True


In [6]:
log_random = pd.read_csv(
    DATA_DIR / "log_random_4_22_to_5_08_pure.csv"
).drop_duplicates().copy()

log_standard = pd.read_csv(
    DATA_DIR / "log_standard_4_22_to_5_08_pure.csv"
).drop_duplicates().copy()

logs = {
    "随机曝光": log_random,
    "正常推荐": log_standard,
}

for name, log in logs.items():
    print("=" * 60)
    print(name)
    print("记录数：", f"{len(log):,}")
    print("日期范围：", log["date"].min(), "-", log["date"].max())
    print("is_rand 分布：", log["is_rand"].value_counts().to_dict())


随机曝光
记录数： 1,186,049
日期范围： 20220422 - 20220508
is_rand 分布： {1: 1186049}
正常推荐
记录数： 289,119
日期范围： 20220422 - 20220508
is_rand 分布： {0: 289119}


In [7]:
for name, log in logs.items():
    log["duration_missing"] = (log["duration_ms"] == 0).astype("int8")
    log["play_ratio"] = pd.NA

    valid_duration = log["duration_ms"] > 0
    log.loc[valid_duration, "play_ratio"] = (
        log.loc[valid_duration, "play_time_ms"]
        / log.loc[valid_duration, "duration_ms"]
    )

    log["event_hour"] = log["hourmin"] // 100


## 4. 核心指标计算函数

下面将指标计算封装为函数，目的是保证不同数据集使用同一套口径。

In [8]:
def calculate_metrics(log: pd.DataFrame, name: str) -> dict:
    exposure_count = len(log)
    active_users = log["user_id"].nunique()
    exposed_videos = log["video_id"].nunique()
    valid_duration = log["duration_ms"] > 0

    return {
        "数据集": name,
        "曝光量": exposure_count,
        "活跃用户数": active_users,
        "曝光视频数": exposed_videos,
        "人均曝光次数": exposure_count / active_users,
        "有效播放反馈率": log["is_click"].mean(),
        "长播率": log["long_view"].mean(),
        "平均播放时长_ms": log["play_time_ms"].mean(),
        "中位数播放比例": log.loc[valid_duration, "play_ratio"].median(),
        "点赞率": log["is_like"].mean(),
        "评论率": log["is_comment"].mean(),
        "关注率": log["is_follow"].mean(),
        "转发率": log["is_forward"].mean(),
        "主页进入率": log["is_profile_enter"].mean(),
        "负反馈率": log["is_hate"].mean(),
        "零视频时长占比": log["duration_missing"].mean(),
    }

metrics = pd.DataFrame([
    calculate_metrics(log_random, "随机曝光"),
    calculate_metrics(log_standard, "正常推荐"),
])

metrics


,数据集,曝光量,活跃用户数,曝光视频数,人均曝光次数,有效播放反馈率,长播率,平均播放时长_ms,中位数播放比例,点赞率,评论率,关注率,转发率,主页进入率,负反馈率,零视频时长占比
0,随机曝光,1186049,27285,7583,43.468902,0.176160,0.084962,6935.569776,0.034799,0.004798,0.000347,0.000261,0.000339,0.005562,0.001139,0.031129
1,正常推荐,289119,25877,6618,11.172818,0.451171,0.318329,21790.113227,0.095467,0.017913,0.002494,0.001307,0.000844,0.018989,0.000785,0.016595


In [9]:
percent_columns = [
    "有效播放反馈率",
    "长播率",
    "中位数播放比例",
    "点赞率",
    "评论率",
    "关注率",
    "转发率",
    "主页进入率",
    "负反馈率",
    "零视频时长占比",
]

count_columns = ["曝光量", "活跃用户数", "曝光视频数"]

metrics_display = metrics.copy()

for column in percent_columns:
    metrics_display[column] = metrics_display[column].map(lambda x: f"{x:.2%}")

for column in count_columns:
    metrics_display[column] = metrics_display[column].map(lambda x: f"{int(x):,}")

metrics_display["人均曝光次数"] = metrics_display["人均曝光次数"].map(lambda x: f"{x:.2f}")
metrics_display["平均播放时长_ms"] = metrics_display["平均播放时长_ms"].map(lambda x: f"{x:,.0f}")

metrics_display


,数据集,曝光量,活跃用户数,曝光视频数,人均曝光次数,有效播放反馈率,长播率,平均播放时长_ms,中位数播放比例,点赞率,评论率,关注率,转发率,主页进入率,负反馈率,零视频时长占比
0,随机曝光,"1,186,049","27,285","7,583",43.47,17.62%,8.50%,"6,936",3.48%,0.48%,0.03%,0.03%,0.03%,0.56%,0.11%,3.11%
1,正常推荐,"289,119","25,877","6,618",11.17,45.12%,31.83%,"21,790",9.55%,1.79%,0.25%,0.13%,0.08%,1.90%,0.08%,1.66%


## 5. 正常推荐相对随机曝光的差异

为了更直观看出差异，下面计算正常推荐相对随机曝光的变化：

```text
绝对差异 = 正常推荐指标 - 随机曝光指标
相对差异 = 绝对差异 / 随机曝光指标
```

注意：这里是描述性差异，不直接等同于严格因果提升。


In [10]:
metric_columns = [
    column for column in metrics.columns
    if column != "数据集"
]

comparison = (
    metrics.set_index("数据集")
    .T
    .rename_axis("指标")
    .reset_index()
)

comparison["绝对差异_正常推荐-随机曝光"] = (
    comparison["正常推荐"] - comparison["随机曝光"]
)
comparison["相对差异"] = (
    comparison["绝对差异_正常推荐-随机曝光"]
    / comparison["随机曝光"]
)

comparison


数据集,指标,随机曝光,正常推荐,绝对差异_正常推荐-随机曝光,相对差异
0,曝光量,1.186049e+06,289119.000000,-896930.000000,-0.756234
1,活跃用户数,2.728500e+04,25877.000000,-1408.000000,-0.051603
2,曝光视频数,7.583000e+03,6618.000000,-965.000000,-0.127258
3,人均曝光次数,4.346890e+01,11.172818,-32.296085,-0.742970
4,有效播放反馈率,1.761597e-01,0.451171,0.275011,1.561146
5,长播率,8.496192e-02,0.318329,0.233367,2.746727
6,平均播放时长_ms,6.935570e+03,21790.113227,14854.543451,2.141791
7,中位数播放比例,3.479924e-02,0.095467,0.060668,1.743369
8,点赞率,4.798284e-03,0.017913,0.013115,2.733218
9,评论率,3.465287e-04,0.002494,0.002147,6.196469


In [11]:
comparison_display = comparison.copy()

rate_metrics = [
    "有效播放反馈率",
    "长播率",
    "中位数播放比例",
    "点赞率",
    "评论率",
    "关注率",
    "转发率",
    "主页进入率",
    "负反馈率",
    "零视频时长占比",
]

for idx, row in comparison_display.iterrows():
    metric_name = row["指标"]

    if metric_name in rate_metrics:
        for column in ["随机曝光", "正常推荐", "绝对差异_正常推荐-随机曝光"]:
            comparison_display.loc[idx, column] = f"{row[column]:.2%}"
        comparison_display.loc[idx, "相对差异"] = f"{row['相对差异']:.2%}"

    elif metric_name in ["曝光量", "活跃用户数", "曝光视频数"]:
        for column in ["随机曝光", "正常推荐", "绝对差异_正常推荐-随机曝光"]:
            comparison_display.loc[idx, column] = f"{row[column]:,.0f}"
        comparison_display.loc[idx, "相对差异"] = f"{row['相对差异']:.2%}"

    else:
        for column in ["随机曝光", "正常推荐", "绝对差异_正常推荐-随机曝光"]:
            comparison_display.loc[idx, column] = f"{row[column]:,.2f}"
        comparison_display.loc[idx, "相对差异"] = f"{row['相对差异']:.2%}"

comparison_display


/var/folders/h5/1fl9gmkx4b56mc3fjtstnq380000gn/T/ipykernel_27159/1540146799.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1,186,049' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  comparison_display.loc[idx, column] = f"{row[column]:,.0f}"
/var/folders/h5/1fl9gmkx4b56mc3fjtstnq380000gn/T/ipykernel_27159/1540146799.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '289,119' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  comparison_display.loc[idx, column] = f"{row[column]:,.0f}"
/var/folders/h5/1fl9gmkx4b56mc3fjtstnq380000gn/T/ipykernel_27159/1540146799.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-896,930' has dtype incompatible with floa

数据集,指标,随机曝光,正常推荐,绝对差异_正常推荐-随机曝光,相对差异
0,曝光量,"1,186,049","289,119","-896,930",-75.62%
1,活跃用户数,"27,285","25,877","-1,408",-5.16%
2,曝光视频数,"7,583","6,618",-965,-12.73%
3,人均曝光次数,43.47,11.17,-32.30,-74.30%
4,有效播放反馈率,17.62%,45.12%,27.50%,156.11%
5,长播率,8.50%,31.83%,23.34%,274.67%
6,平均播放时长_ms,"6,935.57","21,790.11","14,854.54",214.18%
7,中位数播放比例,3.48%,9.55%,6.07%,174.34%
8,点赞率,0.48%,1.79%,1.31%,273.32%
9,评论率,0.03%,0.25%,0.21%,619.65%


### 初步业务解读

从核心指标对比结果看，同期正常推荐在有效播放反馈率、长播率、平均播放时长及互动指标上整体高于随机曝光，说明算法选择的视频整体更符合用户兴趣，能够带来更好的内容消费和互动表现。

同时，正常推荐并非随机选择视频，而是算法主动选择更可能被用户喜欢的内容，因此该差异不能简单解释为严格 A/B 实验中的“算法因果提升”。当前结论属于描述性对比，后续需要结合随机曝光机制进一步分析推荐偏差和因果效果。


## 6. 推荐覆盖与集中度分析

除了看反馈率，还要看推荐是否过度集中在少数视频上。

如果正常推荐效果更好，但主要依靠少数热门视频，可能带来内容生态集中、长尾内容曝光不足等问题。


In [12]:
def top_video_share(log: pd.DataFrame, top_n: int) -> float:
    video_exposures = log["video_id"].value_counts()
    return video_exposures.head(top_n).sum() / len(log)

coverage = pd.DataFrame([
    {
        "数据集": name,
        "曝光视频数": log["video_id"].nunique(),
        "Top10视频曝光占比": top_video_share(log, 10),
        "Top100视频曝光占比": top_video_share(log, 100),
        "Top1000视频曝光占比": top_video_share(log, 1000),
    }
    for name, log in logs.items()
])

coverage_display = coverage.copy()
for column in ["Top10视频曝光占比", "Top100视频曝光占比", "Top1000视频曝光占比"]:
    coverage_display[column] = coverage_display[column].map(lambda x: f"{x:.2%}")
coverage_display["曝光视频数"] = coverage_display["曝光视频数"].map(lambda x: f"{int(x):,}")

coverage_display


,数据集,曝光视频数,Top10视频曝光占比,Top100视频曝光占比,Top1000视频曝光占比
0,随机曝光,"7,583",0.17%,1.66%,15.49%
1,正常推荐,"6,618",4.23%,21.66%,69.54%


### 覆盖与集中度解读

如果正常推荐的 Top 视频曝光占比明显高于随机曝光，说明正常推荐更倾向于集中分发部分视频。这可能是算法选择高质量或高匹配内容的结果，也可能带来内容分发集中化的问题。

后续可以继续按视频类型、视频时长和作者维度拆解，判断推荐集中是否主要由内容质量、作者影响力或用户兴趣匹配造成。


## 7. 本阶段结论

本阶段完成了短视频推荐场景的指标体系设计，并基于同期日志比较了随机曝光与正常推荐的整体表现。

初步结论：

1. 正常推荐在长播、播放时长和互动指标上整体优于随机曝光，说明算法选片具有明显的用户兴趣匹配能力；
2. 负反馈率需要作为护栏指标持续观察，避免推荐策略以牺牲用户体验换取消费时长；
3. 正常推荐与随机曝光的差异属于描述性对比，不能直接等同于新旧算法 A/B 实验的因果提升；
4. 后续分析需要进一步拆解用户、视频、时间和内容维度，并结合随机曝光数据研究推荐偏差。
